## Day 2 — Behavioural Feature Engineering

**Goal:**  
Generate behavioural fingerprints for each account from ~400M transactions.

**Approach**
- Aggregation using **Polars lazy execution**.
- Build behavioural, ratio, and account-level features.

**Key signals**
- Counterparty breadth and transaction volume patterns show strongest behavioural signal.

**Outputs**
- Transaction behavioural features
- Derived ratio features
- Account-level features

In [1]:
import polars as pl
from pathlib import Path
from glob import glob
import warnings
warnings.filterwarnings("ignore")

DATA = Path("../data")
OUTPUTS = Path("../outputs")

# Load what we already have
labels = pl.read_parquet(DATA / "train_labels.parquet")
accounts = pl.read_parquet(DATA / "accounts.parquet")
txn_basic = pl.read_parquet(OUTPUTS / "features_txn_basic.parquet")

print(f"Labels: {labels.shape}")
print(f"Accounts: {accounts.shape}")
print(f"TXN basic features: {txn_basic.shape}")
print(f"\nTXN basic columns: {txn_basic.columns}")

Labels: (96091, 5)
Accounts: (160153, 22)
TXN basic features: (160153, 22)

TXN basic columns: ['account_id', 'total_txn_count', 'avg_amount', 'std_amount', 'total_amount', 'max_amount', 'min_amount', 'reversal_count', 'credit_count', 'debit_count', 'unique_channels', 'unique_counterparties', 'atm_count', 'upi_credit_count', 'upi_debit_count', 'imps_count', 'neft_count', 'standing_instr_count', 'first_txn_date', 'last_txn_date', 'avg_txn_hour', 'late_night_txn_count']


In [2]:
# Check raw timestamp format
sample = pl.read_parquet("../data/transactions/batch-1/part_0001.parquet")
print("Raw timestamp samples:")
print(sample["transaction_timestamp"].head(10))

Raw timestamp samples:
shape: (10,)
Series: 'transaction_timestamp' [str]
[
	"2023-08-31T08:37:05"
	"2023-08-31T11:29:04"
	"2023-08-31T13:31:35"
	"2023-08-31T20:08:21"
	"2023-08-31T20:43:35"
	"2023-09-01T08:51:08"
	"2023-09-01T09:33:39"
	"2023-09-02T03:15:43"
	"2023-09-02T15:24:12"
	"2023-09-02T15:32:48"
]


In [ ]:
from glob import glob

print("Re-running scan with correct timestamp format...")
txn_parts = sorted(glob("../data/transactions/batch-*/part_*.parquet"))
print(f"Total parts: {len(txn_parts)}")

txn_lazy = pl.scan_parquet(txn_parts)

basic_txn_features = (
    txn_lazy
    .with_columns([
        pl.col("transaction_timestamp").str.to_datetime(
            format="%Y-%m-%dT%H:%M:%S", strict=False
        ).alias("ts")
    ])
    .group_by("account_id")
    .agg([
        pl.len().alias("total_txn_count"),
        pl.col("amount").mean().alias("avg_amount"),
        pl.col("amount").std().alias("std_amount"),
        pl.col("amount").sum().alias("total_amount"),
        pl.col("amount").max().alias("max_amount"),
        pl.col("amount").min().alias("min_amount"),
        (pl.col("amount") < 0).sum().alias("reversal_count"),
        (pl.col("txn_type") == "C").sum().alias("credit_count"),
        (pl.col("txn_type") == "D").sum().alias("debit_count"),
        pl.col("channel").n_unique().alias("unique_channels"),
        pl.col("counterparty_id").n_unique().alias("unique_counterparties"),
        (pl.col("channel") == "ATW").sum().alias("atm_count"),
        (pl.col("channel") == "UPC").sum().alias("upi_credit_count"),
        (pl.col("channel") == "UPD").sum().alias("upi_debit_count"),
        (pl.col("channel") == "IPM").sum().alias("imps_count"),
        (pl.col("channel") == "NTD").sum().alias("neft_count"),
        (pl.col("channel") == "STD").sum().alias("standing_instr_count"),
        pl.col("ts").min().alias("first_txn_date"),
        pl.col("ts").max().alias("last_txn_date"),
        pl.col("ts").dt.month().mode().first().alias("peak_month"),
        pl.col("ts").dt.hour().mean().alias("avg_txn_hour"),
        (pl.col("ts").dt.hour() < 6).sum().alias("late_night_txn_count"),
    ])
    .collect()
)

basic_txn_features.write_parquet(OUTPUTS / "features_txn_basic.parquet")
print(f"Done. Shape: {basic_txn_features.shape}")
print(basic_txn_features.select(["account_id", "first_txn_date", "last_txn_date"]).head(5))

Re-running scan with correct timestamp format...
Total parts: 396


In [ ]:
import polars as pl
from pathlib import Path
from glob import glob

DATA = Path("../data")
OUTPUTS = Path("../outputs")

def process_batch(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .with_columns([
            pl.col("transaction_timestamp").str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S", strict=False
            ).alias("ts")
        ])
        .group_by("account_id")
        .agg([
            pl.len().alias("total_txn_count"),
            pl.col("amount").mean().alias("avg_amount"),
            pl.col("amount").std().alias("std_amount"),
            pl.col("amount").sum().alias("total_amount"),
            pl.col("amount").max().alias("max_amount"),
            pl.col("amount").min().alias("min_amount"),
            (pl.col("amount") < 0).sum().alias("reversal_count"),
            (pl.col("txn_type") == "C").sum().alias("credit_count"),
            (pl.col("txn_type") == "D").sum().alias("debit_count"),
            pl.col("channel").n_unique().alias("unique_channels"),
            pl.col("counterparty_id").n_unique().alias("unique_counterparties"),
            (pl.col("channel") == "ATW").sum().alias("atm_count"),
            (pl.col("channel") == "UPC").sum().alias("upi_credit_count"),
            (pl.col("channel") == "UPD").sum().alias("upi_debit_count"),
            (pl.col("channel") == "IPM").sum().alias("imps_count"),
            (pl.col("channel") == "NTD").sum().alias("neft_count"),
            (pl.col("channel") == "STD").sum().alias("standing_instr_count"),
            pl.col("ts").min().alias("first_txn_date"),
            pl.col("ts").max().alias("last_txn_date"),
            pl.col("ts").dt.hour().mean().alias("avg_txn_hour"),
            (pl.col("ts").dt.hour() < 6).sum().alias("late_night_txn_count"),
        ])
        .collect()
    )

# Process batch by batch
batch_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}: {len(parts)} parts...")
    result = process_batch(parts)
    print(f"  Done. Shape: {result.shape}")
    batch_results.append(result)

print("\nCombining batches...")

# Combine all 4 batches — same account may appear in multiple batches
# so we need to aggregate again
combined = pl.concat(batch_results)

final = (
    combined
    .group_by("account_id")
    .agg([
        pl.col("total_txn_count").sum(),
        pl.col("avg_amount").mean(),
        pl.col("std_amount").mean(),
        pl.col("total_amount").sum(),
        pl.col("max_amount").max(),
        pl.col("min_amount").min(),
        pl.col("reversal_count").sum(),
        pl.col("credit_count").sum(),
        pl.col("debit_count").sum(),
        pl.col("unique_channels").max(),
        pl.col("unique_counterparties").max(),
        pl.col("atm_count").sum(),
        pl.col("upi_credit_count").sum(),
        pl.col("upi_debit_count").sum(),
        pl.col("imps_count").sum(),
        pl.col("neft_count").sum(),
        pl.col("standing_instr_count").sum(),
        pl.col("first_txn_date").min(),
        pl.col("last_txn_date").max(),
        pl.col("avg_txn_hour").mean(),
        pl.col("late_night_txn_count").sum(),
    ])
)

final.write_parquet(OUTPUTS / "features_txn_basic.parquet")
print(f"\nFinal shape: {final.shape}")
print(final.head(3))

Processing batch-1: 100 parts...
  Done. Shape: (40076, 22)
Processing batch-2: 100 parts...
  Done. Shape: (39662, 22)
Processing batch-3: 100 parts...
  Done. Shape: (39446, 22)
Processing batch-4: 96 parts...
  Done. Shape: (49269, 22)

Combining batches...

Final shape: (160153, 22)
shape: (3, 22)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ account_i ┆ total_txn ┆ avg_amoun ┆ std_amoun ┆ … ┆ first_txn ┆ last_txn_ ┆ avg_txn_h ┆ late_nig │
│ d         ┆ _count    ┆ t         ┆ t         ┆   ┆ _date     ┆ date      ┆ our       ┆ ht_txn_c │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ount     │
│ str       ┆ u32       ┆ f64       ┆ f64       ┆   ┆ datetime[ ┆ datetime[ ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ μs]       ┆ μs]       ┆           ┆ u32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══

In [ ]:
txn_basic = pl.read_parquet(OUTPUTS / "features_txn_basic.parquet")

txn_derived = txn_basic.with_columns([
    # Pass-through proxy: credit and debit both high relative to total
    (pl.col("credit_count") / pl.col("total_txn_count")).alias("credit_ratio"),
    (pl.col("debit_count") / pl.col("total_txn_count")).alias("debit_ratio"),
    
    # ATM rate
    (pl.col("atm_count") / pl.col("total_txn_count")).alias("atm_rate"),
    
    # UPI dominance
    ((pl.col("upi_credit_count") + pl.col("upi_debit_count")) / 
     pl.col("total_txn_count")).alias("upi_rate"),
    
    # IMPS + NEFT rate (interbank = higher value transfers)
    ((pl.col("imps_count") + pl.col("neft_count")) / 
     pl.col("total_txn_count")).alias("interbank_rate"),
    
    # Standing instruction absence (mules have no recurring payments)
    (pl.col("standing_instr_count") / pl.col("total_txn_count")).alias("standing_instr_rate"),
    
    # Reversal rate
    (pl.col("reversal_count") / pl.col("total_txn_count")).alias("reversal_rate"),
    
    # Amount volatility — high std relative to mean = erratic amounts
    (pl.col("std_amount") / (pl.col("avg_amount").abs() + 1)).alias("amount_volatility"),
    
    # Active span in days
    ((pl.col("last_txn_date") - pl.col("first_txn_date"))
     .dt.total_days()).alias("active_span_days"),
    
    # Transaction velocity — txns per active day
    (pl.col("total_txn_count") / 
     ((pl.col("last_txn_date") - pl.col("first_txn_date"))
      .dt.total_days() + 1)).alias("txn_velocity"),
    
    # Late night rate
    (pl.col("late_night_txn_count") / pl.col("total_txn_count")).alias("late_night_rate"),
])

print(f"Shape: {txn_derived.shape}")
print(f"New columns added: {[c for c in txn_derived.columns if c not in txn_basic.columns]}")
txn_derived.write_parquet(OUTPUTS / "features_txn_derived.parquet")
print("Saved.")

Shape: (160153, 33)
New columns added: ['credit_ratio', 'debit_ratio', 'atm_rate', 'upi_rate', 'interbank_rate', 'standing_instr_rate', 'reversal_rate', 'amount_volatility', 'active_span_days', 'txn_velocity', 'late_night_rate']
Saved.


In [ ]:
print("Computing pass-through rate per batch...")

def compute_passthrough(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .with_columns([
            pl.col("transaction_timestamp").str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S", strict=False
            ).alias("ts")
        ])
        .with_columns([
            pl.col("ts").dt.date().alias("date")
        ])
        .group_by(["account_id", "date"])
        .agg([
            ((pl.col("txn_type") == "C").sum() > 0).alias("has_credit"),
            ((pl.col("txn_type") == "D").sum() > 0).alias("has_debit"),
        ])
        .with_columns([
            (pl.col("has_credit") & pl.col("has_debit")).alias("is_passthrough_day")
        ])
        .group_by("account_id")
        .agg([
            pl.col("is_passthrough_day").sum().alias("passthrough_days"),
            pl.len().alias("active_days"),
        ])
        .collect()
    )

pt_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = compute_passthrough(parts)
    pt_results.append(result)
    print(f"  Done: {result.shape}")

# Combine
pt_combined = pl.concat(pt_results)
passthrough = (
    pt_combined
    .group_by("account_id")
    .agg([
        pl.col("passthrough_days").sum(),
        pl.col("active_days").sum(),
    ])
    .with_columns([
        (pl.col("passthrough_days") / pl.col("active_days")).alias("passthrough_rate")
    ])
)

passthrough.write_parquet(OUTPUTS / "features_passthrough.parquet")
print(f"\nDone. Shape: {passthrough.shape}")
print(passthrough.head(5))

Computing pass-through rate per batch...
Processing batch-1...
  Done: (40076, 3)
Processing batch-2...
  Done: (39662, 3)
Processing batch-3...
  Done: (39446, 3)
Processing batch-4...
  Done: (49269, 3)

Done. Shape: (160153, 4)
shape: (5, 4)
┌─────────────┬──────────────────┬─────────────┬──────────────────┐
│ account_id  ┆ passthrough_days ┆ active_days ┆ passthrough_rate │
│ ---         ┆ ---              ┆ ---         ┆ ---              │
│ str         ┆ u32              ┆ u32         ┆ f64              │
╞═════════════╪══════════════════╪═════════════╪══════════════════╡
│ ACCT_160785 ┆ 698              ┆ 855         ┆ 0.816374         │
│ ACCT_086811 ┆ 211              ┆ 489         ┆ 0.431493         │
│ ACCT_133170 ┆ 438              ┆ 764         ┆ 0.573298         │
│ ACCT_059398 ┆ 146              ┆ 699         ┆ 0.20887          │
│ ACCT_116284 ┆ 286              ┆ 288         ┆ 0.993056         │
└─────────────┴──────────────────┴─────────────┴──────────────────┘


In [ ]:
print("Computing burst ratio per batch...")

def compute_burst(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .with_columns([
            pl.col("transaction_timestamp").str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S", strict=False
            ).alias("ts")
        ])
        .with_columns([
            pl.col("ts").dt.year().alias("year"),
            pl.col("ts").dt.month().alias("month"),
        ])
        .group_by(["account_id", "year", "month"])
        .agg(pl.len().alias("monthly_txn_count"))
        .collect()
    )

burst_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = compute_burst(parts)
    burst_results.append(result)
    print(f"  Done: {result.shape}")

# Combine and compute burst ratio
monthly = (
    pl.concat(burst_results)
    .group_by(["account_id", "year", "month"])
    .agg(pl.col("monthly_txn_count").sum())
)

burst = (
    monthly
    .group_by("account_id")
    .agg([
        pl.col("monthly_txn_count").max().alias("peak_monthly_txn"),
        pl.col("monthly_txn_count").median().alias("median_monthly_txn"),
        pl.col("monthly_txn_count").std().alias("std_monthly_txn"),
        pl.col("monthly_txn_count").count().alias("active_months"),
    ])
    .with_columns([
        (pl.col("peak_monthly_txn") / 
         (pl.col("median_monthly_txn") + 1)).alias("burst_ratio")
    ])
)

burst.write_parquet(OUTPUTS / "features_burst.parquet")
print(f"\nDone. Shape: {burst.shape}")
print(burst.head(5))

Computing burst ratio per batch...
Processing batch-1...
  Done: (1269222, 4)
Processing batch-2...
  Done: (1258924, 4)
Processing batch-3...
  Done: (1252698, 4)
Processing batch-4...
  Done: (1272957, 4)

Done. Shape: (160153, 6)
shape: (5, 6)
┌─────────────┬──────────────────┬─────────────────┬─────────────────┬───────────────┬─────────────┐
│ account_id  ┆ peak_monthly_txn ┆ median_monthly_ ┆ std_monthly_txn ┆ active_months ┆ burst_ratio │
│ ---         ┆ ---              ┆ txn             ┆ ---             ┆ ---           ┆ ---         │
│ str         ┆ u32              ┆ ---             ┆ f64             ┆ u32           ┆ f64         │
│             ┆                  ┆ f64             ┆                 ┆               ┆             │
╞═════════════╪══════════════════╪═════════════════╪═════════════════╪═══════════════╪═════════════╡
│ ACCT_052956 ┆ 66               ┆ 51.0            ┆ 11.249058       ┆ 27            ┆ 1.269231    │
│ ACCT_039070 ┆ 19               ┆ 12.0       

In [ ]:
labels = pl.read_parquet(DATA / "train_labels.parquet")
passthrough = pl.read_parquet(OUTPUTS / "features_passthrough.parquet")
burst = pl.read_parquet(OUTPUTS / "features_burst.parquet")
txn_derived = pl.read_parquet(OUTPUTS / "features_txn_derived.parquet")

# Join everything with labels
check = (
    labels
    .join(passthrough, on="account_id", how="left")
    .join(burst, on="account_id", how="left")
    .join(txn_derived.select(["account_id", "atm_rate", "amount_volatility", 
                               "txn_velocity", "late_night_rate"]), 
          on="account_id", how="left")
)

# Compare mule vs legit medians
mule_check = check.filter(pl.col("is_mule") == 1)
legit_check = check.filter(pl.col("is_mule") == 0)

features_to_check = ["passthrough_rate", "burst_ratio", "atm_rate", 
                      "amount_volatility", "txn_velocity", "late_night_rate"]

print(f"{'Feature':<25} {'Mule Median':>12} {'Legit Median':>13} {'Ratio':>8}")
print("-" * 62)
for feat in features_to_check:
    m = mule_check[feat].median()
    l = legit_check[feat].median()
    ratio = m/l if l and l != 0 else float('inf')
    print(f"{feat:<25} {m:>12.4f} {l:>13.4f} {ratio:>8.2f}x")

Feature                    Mule Median  Legit Median    Ratio
--------------------------------------------------------------
passthrough_rate                0.5252        0.5378     0.98x
burst_ratio                     1.2593        1.2378     1.02x
atm_rate                        0.0068        0.0064     1.06x
amount_volatility               6.3690        7.5930     0.84x
txn_velocity                    1.9788        2.3049     0.86x
late_night_rate                 0.0445        0.0445     1.00x


In [ ]:
# Join all txn features with labels
check_full = (
    labels
    .join(txn_derived, on="account_id", how="left")
    .join(passthrough, on="account_id", how="left")
    .join(burst, on="account_id", how="left")
)

mule_check = check_full.filter(pl.col("is_mule") == 1)
legit_check = check_full.filter(pl.col("is_mule") == 0)

# Check ALL features
all_features = [c for c in check_full.columns 
                if c not in ["account_id", "is_mule", "mule_flag_date", 
                             "alert_reason", "flagged_by_branch",
                             "first_txn_date", "last_txn_date"]]

results = []
for feat in all_features:
    try:
        m = mule_check[feat].median()
        l = legit_check[feat].median()
        if l and abs(l) > 0.0001:
            ratio = abs(m/l)
        else:
            ratio = 0
        results.append({"feature": feat, "mule": round(m, 4), 
                        "legit": round(l, 4), "ratio": round(ratio, 3)})
    except:
        pass

results_df = pl.DataFrame(results).sort("ratio", descending=True)
print(results_df)

shape: (38, 4)
┌───────────────────────┬─────────────┬─────────────┬───────┐
│ feature               ┆ mule        ┆ legit       ┆ ratio │
│ ---                   ┆ ---         ┆ ---         ┆ ---   │
│ str                   ┆ f64         ┆ f64         ┆ f64   │
╞═══════════════════════╪═════════════╪═════════════╪═══════╡
│ unique_counterparties ┆ 37.0        ┆ 18.0        ┆ 2.056 │
│ std_monthly_txn       ┆ 14.9665     ┆ 10.7909     ┆ 1.387 │
│ interbank_rate        ┆ 0.061       ┆ 0.0557      ┆ 1.095 │
│ atm_rate              ┆ 0.0068      ┆ 0.0064      ┆ 1.06  │
│ peak_monthly_txn      ┆ 90.0        ┆ 88.0        ┆ 1.023 │
│ …                     ┆ …           ┆ …           ┆ …     │
│ std_amount            ┆ 148319.5248 ┆ 190253.1999 ┆ 0.78  │
│ upi_debit_count       ┆ 541.0       ┆ 694.0       ┆ 0.78  │
│ passthrough_days      ┆ 247.0       ┆ 340.0       ┆ 0.726 │
│ reversal_count        ┆ 7.0         ┆ 10.0        ┆ 0.7   │
│ min_amount            ┆ -33000.0    ┆ -51280.215  ┆ 0

In [ ]:
accounts = pl.read_parquet(DATA / "accounts.parquet")
customers = pl.read_parquet(DATA / "customers.parquet")
linkage = pl.read_parquet(DATA / "customer_account_linkage.parquet")

# Join accounts with labels
acct_check = (
    labels
    .join(accounts, on="account_id", how="left")
    .join(linkage, on="account_id", how="left")
    .join(customers, on="customer_id", how="left")
)

mule_acct = acct_check.filter(pl.col("is_mule") == 1)
legit_acct = acct_check.filter(pl.col("is_mule") == 0)

# Account status — frozen rate
print("=== Account Status ===")
print("Mule frozen rate:", 
      (mule_acct["account_status"] == "frozen").mean())
print("Legit frozen rate:", 
      (legit_acct["account_status"] == "frozen").mean())

# Account age
print("\n=== Account Age ===")
acct_check2 = acct_check.with_columns([
    (pl.lit("2025-06-30").str.to_date() - 
     pl.col("account_opening_date").str.to_date()).dt.total_days().alias("account_age_days")
])
m_age = acct_check2.filter(pl.col("is_mule")==1)["account_age_days"].median()
l_age = acct_check2.filter(pl.col("is_mule")==0)["account_age_days"].median()
print(f"Mule median age: {m_age:.0f} days")
print(f"Legit median age: {l_age:.0f} days")

# Rural branch
print("\n=== Rural Branch ===")
print("Mule rural rate:", mule_acct["rural_branch"].mean())
print("Legit rural rate:", legit_acct["rural_branch"].mean())

# KYC compliant
print("\n=== KYC Compliant ===")
print("Mule KYC rate:", (mule_acct["kyc_compliant"] == "Y").mean())
print("Legit KYC rate:", (legit_acct["kyc_compliant"] == "Y").mean())

# Multiple accounts per customer
print("\n=== Multiple Accounts ===")
acct_per_customer = (
    linkage
    .group_by("customer_id")
    .agg(pl.len().alias("num_accounts"))
)
multi_acct = (
    acct_check
    .join(acct_per_customer, on="customer_id", how="left")
)
m_multi = multi_acct.filter(pl.col("is_mule")==1)["num_accounts"].median()
l_multi = multi_acct.filter(pl.col("is_mule")==0)["num_accounts"].median()
print(f"Mule median accounts per customer: {m_multi}")
print(f"Legit median accounts per customer: {l_multi}")

=== Account Status ===
Mule frozen rate: 0.4912411479686918
Legit frozen rate: 0.04493191161356629

=== Account Age ===
Mule median age: 759 days
Legit median age: 803 days

=== Rural Branch ===
Mule rural rate: None
Legit rural rate: None

=== KYC Compliant ===
Mule KYC rate: 0.90831159150205
Legit KYC rate: 0.8988951695786228

=== Multiple Accounts ===
Mule median accounts per customer: 1.0
Legit median accounts per customer: 1.0


In [ ]:
def compute_entropy(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .group_by(["account_id", "counterparty_id"])
        .agg([
            pl.len().alias("txn_count"),
            pl.col("amount").abs().sum().alias("volume"),
        ])
        .collect()
    )

print("Computing counterparty entropy per batch...")
entropy_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = compute_entropy(parts)
    entropy_results.append(result)
    print(f"  Done: {result.shape}")

# Combine across batches
print("\nCombining...")
cp_combined = (
    pl.concat(entropy_results)
    .group_by(["account_id", "counterparty_id"])
    .agg([
        pl.col("txn_count").sum(),
        pl.col("volume").sum(),
    ])
)

# Compute entropy and concentration per account
print("Computing entropy...")
account_entropy = (
    cp_combined
    .with_columns([
        pl.col("volume") / 
        pl.col("volume").sum().over("account_id")
    ].alias("vol_share"))
    .with_columns([
        (-pl.col("vol_share") * (pl.col("vol_share") + 1e-10).log()).alias("entropy_contrib")
    ])
    .group_by("account_id")
    .agg([
        pl.col("entropy_contrib").sum().alias("counterparty_entropy"),
        pl.col("txn_count").sum().alias("total_cp_txns"),
        pl.len().alias("unique_counterparties_check"),
        # Volume concentration — top 3 counterparties
        (pl.col("volume").sort(descending=True).head(3).sum() / 
         pl.col("volume").sum()).alias("pct_volume_top3"),
    ])
)

account_entropy.write_parquet(OUTPUTS / "features_entropy.parquet")
print(f"Done. Shape: {account_entropy.shape}")
print(account_entropy.head(3))

Computing counterparty entropy per batch...
Processing batch-1...
  Done: (711777, 4)
Processing batch-2...
  Done: (707310, 4)
Processing batch-3...
  Done: (702703, 4)
Processing batch-4...
  Done: (1133113, 4)

Combining...
Computing entropy...


AttributeError: 'list' object has no attribute 'alias'

In [ ]:
def compute_entropy(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .group_by(["account_id", "counterparty_id"])
        .agg([
            pl.len().alias("txn_count"),
            pl.col("amount").abs().sum().alias("volume"),
        ])
        .collect()
    )

print("Computing counterparty entropy per batch...")
entropy_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = compute_entropy(parts)
    entropy_results.append(result)
    print(f"  Done: {result.shape}")

print("\nCombining...")
cp_combined = (
    pl.concat(entropy_results)
    .group_by(["account_id", "counterparty_id"])
    .agg([
        pl.col("txn_count").sum(),
        pl.col("volume").sum(),
    ])
)

# Total volume per account
account_total = (
    cp_combined
    .group_by("account_id")
    .agg(pl.col("volume").sum().alias("total_volume"))
)

# Join back and compute share
cp_with_share = (
    cp_combined
    .join(account_total, on="account_id")
    .with_columns([
        (pl.col("volume") / pl.col("total_volume")).alias("vol_share")
    ])
    .with_columns([
        (-pl.col("vol_share") * (pl.col("vol_share") + 1e-10).log()).alias("entropy_contrib")
    ])
)

# Top 3 volume per account
top3 = (
    cp_combined
    .sort(["account_id", "volume"], descending=[False, True])
    .group_by("account_id")
    .agg([
        pl.col("volume").head(3).sum().alias("top3_volume"),
        pl.col("volume").sum().alias("total_vol"),
    ])
    .with_columns([
        (pl.col("top3_volume") / pl.col("total_vol")).alias("pct_volume_top3")
    ])
)

# Final entropy per account
account_entropy = (
    cp_with_share
    .group_by("account_id")
    .agg([
        pl.col("entropy_contrib").sum().alias("counterparty_entropy"),
        pl.len().alias("unique_cp_count"),
    ])
    .join(top3.select(["account_id", "pct_volume_top3"]), on="account_id")
)

account_entropy.write_parquet(OUTPUTS / "features_entropy.parquet")
print(f"Done. Shape: {account_entropy.shape}")

# Quick signal check
check_e = (
    labels
    .join(account_entropy, on="account_id", how="left")
)
mule_e = check_e.filter(pl.col("is_mule") == 1)
legit_e = check_e.filter(pl.col("is_mule") == 0)

print(f"\nCounterparty entropy  — Mule: {mule_e['counterparty_entropy'].median():.3f} | Legit: {legit_e['counterparty_entropy'].median():.3f}")
print(f"Unique counterparties — Mule: {mule_e['unique_cp_count'].median():.1f} | Legit: {legit_e['unique_cp_count'].median():.1f}")
print(f"Pct volume top3       — Mule: {mule_e['pct_volume_top3'].median():.3f} | Legit: {legit_e['pct_volume_top3'].median():.3f}")

Computing counterparty entropy per batch...
Processing batch-1...
  Done: (711777, 4)
Processing batch-2...
  Done: (707310, 4)
Processing batch-3...
  Done: (702703, 4)
Processing batch-4...
  Done: (1133113, 4)

Combining...
Done. Shape: (160153, 4)

Counterparty entropy  — Mule: 2.545 | Legit: 2.455
Unique counterparties — Mule: 45.0 | Legit: 18.0
Pct volume top3       — Mule: 0.457 | Legit: 0.454


In [ ]:
print("Building mule counterparty contamination score...")

# Step 1: Get all counterparties of confirmed mules from training data
mule_ids = labels.filter(pl.col("is_mule") == 1)["account_id"].to_list()
print(f"Known mules: {len(mule_ids)}")

# Collect all counterparties that mules transact with
def get_mule_counterparties(batch_parts, mule_ids):
    return (
        pl.scan_parquet(batch_parts)
        .filter(pl.col("account_id").is_in(mule_ids))
        .select(["account_id", "counterparty_id"])
        .collect()
    )

mule_cp_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = get_mule_counterparties(parts, mule_ids)
    mule_cp_results.append(result)
    print(f"  Mule transactions found: {result.shape}")

mule_cp = pl.concat(mule_cp_results)

# Unique counterparties per mule account
mule_counterparty_set = (
    mule_cp
    .group_by("counterparty_id")
    .agg(pl.col("account_id").n_unique().alias("mule_account_count"))
)

print(f"\nUnique counterparties in mule network: {len(mule_counterparty_set)}")
print(f"Top 10 most shared counterparties:")
print(mule_counterparty_set.sort("mule_account_count", descending=True).head(10))

Building mule counterparty contamination score...
Known mules: 2683
Processing batch-1...
  Mule transactions found: (1252373, 2)
Processing batch-2...
  Mule transactions found: (1320627, 2)
Processing batch-3...
  Mule transactions found: (1286270, 2)
Processing batch-4...
  Mule transactions found: (1384656, 2)

Unique counterparties in mule network: 73408
Top 10 most shared counterparties:
shape: (10, 2)
┌─────────────────┬────────────────────┐
│ counterparty_id ┆ mule_account_count │
│ ---             ┆ ---                │
│ str             ┆ u32                │
╞═════════════════╪════════════════════╡
│ BANK_INTEREST   ┆ 2465               │
│ CP_BR4091_002   ┆ 26                 │
│ CP_BR4091_000   ┆ 25                 │
│ CP_BR4091_001   ┆ 25                 │
│ CP_BR1523_001   ┆ 21                 │
│ CP_BR1523_002   ┆ 21                 │
│ CP_BR8103_002   ┆ 20                 │
│ CP_BR8103_001   ┆ 20                 │
│ CP_BR1523_000   ┆ 19                 │
│ CP_BR4091_00

In [ ]:
# Exclude bank-generated counterparties — they appear everywhere and have no signal
exclude_counterparties = ["BANK_INTEREST"]

# Also exclude any counterparty appearing in >500 mule accounts
# These are system-wide counterparties (bank fees, interest, etc.) not real connections
high_freq_cp = (
    mule_counterparty_set
    .filter(pl.col("mule_account_count") > 500)
    ["counterparty_id"].to_list()
)
print(f"High frequency counterparties to exclude: {high_freq_cp}")

# Clean mule counterparty set
mule_cp_clean = mule_counterparty_set.filter(
    ~pl.col("counterparty_id").is_in(high_freq_cp)
)
print(f"Clean mule counterparty network size: {len(mule_cp_clean)}")

# These are the meaningful mule-connected counterparties
mule_cp_list = mule_cp_clean["counterparty_id"].to_list()
print(f"Counterparties connected to 2+ mule accounts: "
      f"{len(mule_cp_clean.filter(pl.col('mule_account_count') >= 2))}")

High frequency counterparties to exclude: ['BANK_INTEREST']
Clean mule counterparty network size: 73407
Counterparties connected to 2+ mule accounts: 34164


In [ ]:
def get_account_counterparties(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .select(["account_id", "counterparty_id"])
        .collect()
    )

print("Getting all account counterparties...")
all_cp_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = get_account_counterparties(parts)
    all_cp_results.append(result)
    print(f"  Done: {result.shape}")

all_cp = (
    pl.concat(all_cp_results)
    .unique(["account_id", "counterparty_id"])
)
print(f"\nTotal unique account-counterparty pairs: {len(all_cp)}")

# Join with mule counterparty scores
contamination = (
    all_cp
    .join(mule_cp_clean, on="counterparty_id", how="left")
    .with_columns([
        pl.col("mule_account_count").fill_null(0)
    ])
    .group_by("account_id")
    .agg([
        # How many of this account's counterparties are in the mule network
        (pl.col("mule_account_count") > 0).sum().alias("mule_network_cp_count"),
        # Total unique counterparties
        pl.len().alias("total_cp_count"),
        # Weighted score — counterparties shared by more mules = higher weight
        pl.col("mule_account_count").sum().alias("mule_cp_weighted_score"),
        # Max mule connections through any single counterparty
        pl.col("mule_account_count").max().alias("max_mule_cp_connection"),
    ])
    .with_columns([
        # Contamination rate — what % of counterparties are mule-connected
        (pl.col("mule_network_cp_count") / pl.col("total_cp_count"))
        .alias("contamination_rate")
    ])
)

contamination.write_parquet(OUTPUTS / "features_contamination.parquet")
print(f"Done. Shape: {contamination.shape}")

# Signal check
check_c = labels.join(contamination, on="account_id", how="left")
mule_c = check_c.filter(pl.col("is_mule") == 1)
legit_c = check_c.filter(pl.col("is_mule") == 0)

print(f"\n{'Feature':<30} {'Mule':>10} {'Legit':>10} {'Ratio':>8}")
print("-" * 62)
for feat in ["mule_network_cp_count", "contamination_rate", 
             "mule_cp_weighted_score", "max_mule_cp_connection"]:
    m = mule_c[feat].median()
    l = legit_c[feat].median()
    ratio = m/l if l and l > 0 else float('inf')
    print(f"{feat:<30} {m:>10.3f} {l:>10.3f} {ratio:>8.2f}x")

Getting all account counterparties...
Processing batch-1...
  Done: (100686033, 2)
Processing batch-2...
  Done: (100407662, 2)
Processing batch-3...
  Done: (100460390, 2)
Processing batch-4...
  Done: (96321238, 2)

Total unique account-counterparty pairs: 3254788
Done. Shape: (160153, 6)

Feature                              Mule      Legit    Ratio
--------------------------------------------------------------
mule_network_cp_count              44.000     11.000     4.00x
contamination_rate                  0.980      0.654     1.50x
mule_cp_weighted_score             99.000     19.000     5.21x
max_mule_cp_connection              5.000      3.000     1.67x


In [ ]:
accounts = pl.read_parquet(DATA / "accounts.parquet")
customers = pl.read_parquet(DATA / "customers.parquet")
linkage = pl.read_parquet(DATA / "customer_account_linkage.parquet")
demo = pl.read_parquet(DATA / "demographics.parquet")
branch = pl.read_parquet(DATA / "branch.parquet")
txn_basic = pl.read_parquet(OUTPUTS / "features_txn_basic.parquet")

# Reference date = last date in dataset
REF_DATE = pl.lit("2025-06-30").str.to_date()

account_features = (
    accounts
    .with_columns([
        # Account age
        (REF_DATE - pl.col("account_opening_date").str.to_date())
        .dt.total_days().alias("account_age_days"),
        
        # Frozen flag
        (pl.col("account_status") == "frozen").cast(pl.Int8).alias("is_frozen"),
        
        # Mobile update recency — days since last mobile update
        (REF_DATE - pl.col("last_mobile_update_date").str.to_date())
        .dt.total_days().alias("days_since_mobile_update"),
        
        # Had mobile update at all
        pl.col("last_mobile_update_date").is_not_null()
        .cast(pl.Int8).alias("had_mobile_update"),
        
        # Balance inconsistency — monthly vs quarterly vs daily avg
        (pl.col("monthly_avg_balance") - pl.col("quarterly_avg_balance"))
        .abs().alias("balance_monthly_quarterly_diff"),
        
        (pl.col("daily_avg_balance") - pl.col("monthly_avg_balance"))
        .abs().alias("balance_daily_monthly_diff"),
        
        # Freeze duration
        (pl.col("unfreeze_date").str.to_date() - pl.col("freeze_date").str.to_date())
        .dt.total_days().alias("freeze_duration_days"),
        
        # KYC compliant flag
        (pl.col("kyc_compliant") == "Y").cast(pl.Int8).alias("kyc_flag"),
        
        # Nomination flag
        (pl.col("nomination_flag") == "Y").cast(pl.Int8).alias("has_nomination"),
        
        # Cheque availed
        (pl.col("cheque_availed") == "Y").cast(pl.Int8).alias("has_cheque"),
    ])
    .select([
        "account_id", "branch_code", "account_age_days", "is_frozen",
        "days_since_mobile_update", "had_mobile_update",
        "balance_monthly_quarterly_diff", "balance_daily_monthly_diff",
        "freeze_duration_days", "kyc_flag", "has_nomination",
        "has_cheque", "avg_balance", "monthly_avg_balance",
        "product_family", "rural_branch"
    ])
)

# Income mismatch — avg balance vs transaction volume
account_features = (
    account_features
    .join(txn_basic.select(["account_id", "total_amount", "avg_amount"]), 
          on="account_id", how="left")
    .with_columns([
        # Balance to transaction ratio — mules have low balance relative to txn volume
        (pl.col("avg_balance") / (pl.col("avg_amount").abs() + 1))
        .alias("balance_to_txn_ratio"),
        
        # Income mismatch — total volume vs avg balance
        (pl.col("total_amount").abs() / (pl.col("avg_balance").abs() + 1))
        .alias("volume_to_balance_ratio"),
    ])
)

print(f"Account features shape: {account_features.shape}")
print(account_features.columns)

Account features shape: (160153, 20)
['account_id', 'branch_code', 'account_age_days', 'is_frozen', 'days_since_mobile_update', 'had_mobile_update', 'balance_monthly_quarterly_diff', 'balance_daily_monthly_diff', 'freeze_duration_days', 'kyc_flag', 'has_nomination', 'has_cheque', 'avg_balance', 'monthly_avg_balance', 'product_family', 'rural_branch', 'total_amount', 'avg_amount', 'balance_to_txn_ratio', 'volume_to_balance_ratio']


In [ ]:
# Branch mule concentration
branch_mule = (
    labels
    .join(accounts.select(["account_id", "branch_code"]), 
          on="account_id", how="left")
    .group_by("branch_code")
    .agg([
        pl.len().alias("branch_total_accounts"),
        pl.col("is_mule").sum().alias("branch_mule_count"),
        pl.col("is_mule").mean().alias("branch_mule_rate"),
    ])
)

global_mule_rate = labels["is_mule"].mean()

branch_features = (
    branch_mule
    .with_columns([
        (pl.col("branch_mule_rate") / global_mule_rate)
        .alias("branch_relative_risk"),
    ])
    .join(branch, on="branch_code", how="left")
    .with_columns([
        # Mules per employee — normalized branch risk
        (pl.col("branch_mule_count") / 
         (pl.col("branch_employee_count") + 1))
        .alias("mules_per_employee"),
        
        # Branch type encoding
        pl.col("branch_type").replace(
            {"urban": 2, "semi-urban": 1, "rural": 0}
        ).alias("branch_type_encoded"),
    ])
    .select([
        "branch_code", "branch_mule_rate", "branch_relative_risk",
        "branch_mule_count", "branch_total_accounts",
        "branch_employee_count", "mules_per_employee",
        "branch_type_encoded", "branch_turnover", "branch_asset_size"
    ])
)

print(f"Branch features shape: {branch_features.shape}")
print(f"\nTop 10 highest risk branches:")
print(branch_features.sort("branch_relative_risk", descending=True).head(10))

Branch features shape: (9000, 10)

Top 10 highest risk branches:
shape: (10, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ branch_co ┆ branch_mu ┆ branch_re ┆ branch_mu ┆ … ┆ mules_per ┆ branch_ty ┆ branch_tu ┆ branch_a │
│ de        ┆ le_rate   ┆ lative_ri ┆ le_count  ┆   ┆ _employee ┆ pe_encode ┆ rnover    ┆ sset_siz │
│ ---       ┆ ---       ┆ sk        ┆ ---       ┆   ┆ ---       ┆ d         ┆ ---       ┆ e        │
│ i64       ┆ f64       ┆ ---       ┆ i64       ┆   ┆ f64       ┆ ---       ┆ f64       ┆ ---      │
│           ┆           ┆ f64       ┆           ┆   ┆           ┆ str       ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1523      ┆ 0.581395  ┆ 20.822535 ┆ 25        ┆ … ┆ 12.5      ┆ 0         ┆ 193.71    ┆ 26.29    │
│ 4091      ┆ 0.576923  ┆ 20.662361 ┆ 30        ┆ … ┆ 10.0      ┆ 0         ┆ 225.88    ┆ 29.58    │
│ 8103    

In [ ]:
demo_features = (
    demo
    .with_columns([
        # Address update recency
        (REF_DATE - pl.col("address_last_update_date").str.to_date())
        .dt.total_days().alias("days_since_address_update"),
        
        # Passbook update recency  
        (REF_DATE - pl.col("passbook_last_update_date").str.to_date())
        .dt.total_days().alias("days_since_passbook_update"),
        
        # Joint account flag
        (pl.col("joint_account_flag") == "Y").cast(pl.Int8).alias("is_joint_account"),
        
        # NRI flag
        (pl.col("nri_flag") == "Y").cast(pl.Int8).alias("is_nri"),
        
        # Gender encoding
        (pl.col("gender") == "M").cast(pl.Int8).alias("is_male"),
    ])
    .select([
        "customer_id", "days_since_address_update", 
        "days_since_passbook_update", "is_joint_account", 
        "is_nri", "is_male"
    ])
)

# Customer age from customers table
customer_features = (
    customers
    .with_columns([
        (REF_DATE - pl.col("date_of_birth").str.to_date())
        .dt.total_days().alias("customer_age_days"),
        
        (REF_DATE - pl.col("relationship_start_date").str.to_date())
        .dt.total_days().alias("relationship_tenure_days"),
        
        # KYC score
        ((pl.col("pan_available") == "Y").cast(pl.Int8) +
         (pl.col("aadhaar_available") == "Y").cast(pl.Int8) +
         (pl.col("passport_available") == "Y").cast(pl.Int8))
        .alias("kyc_doc_score"),
        
        # Digital banking score
        ((pl.col("mobile_banking_flag") == "Y").cast(pl.Int8) +
         (pl.col("internet_banking_flag") == "Y").cast(pl.Int8) +
         (pl.col("atm_card_flag") == "Y").cast(pl.Int8))
        .alias("digital_banking_score"),
    ])
    .select([
        "customer_id", "customer_age_days", "relationship_tenure_days",
        "kyc_doc_score", "digital_banking_score"
    ])
    .join(demo_features, on="customer_id", how="left")
)

print(f"Customer features shape: {customer_features.shape}")

Customer features shape: (159416, 10)


In [ ]:
# Save individual feature files
account_features.write_parquet(OUTPUTS / "features_account.parquet")
branch_features.write_parquet(OUTPUTS / "features_branch.parquet")
customer_features.write_parquet(OUTPUTS / "features_customer.parquet")
print("All feature files saved.")

# Now join everything and check signals
master = (
    labels
    .join(account_features, on="account_id", how="left")
    .join(
        accounts.select(["account_id", "branch_code"]),
        on="account_id", how="left"
    )
    .join(branch_features, on="branch_code", how="left")
    .join(linkage, on="account_id", how="left")
    .join(customer_features, on="customer_id", how="left")
)

mule_m = master.filter(pl.col("is_mule") == 1)
legit_m = master.filter(pl.col("is_mule") == 0)

check_features = [
    "is_frozen", "account_age_days", "had_mobile_update",
    "days_since_mobile_update", "balance_to_txn_ratio",
    "volume_to_balance_ratio", "branch_relative_risk",
    "mules_per_employee", "kyc_doc_score", "digital_banking_score",
    "is_joint_account", "is_nri", "customer_age_days",
    "balance_monthly_quarterly_diff"
]

print(f"\n{'Feature':<35} {'Mule':>10} {'Legit':>10} {'Ratio':>8}")
print("-" * 67)
for feat in check_features:
    try:
        m = mule_m[feat].median()
        l = legit_m[feat].median()
        if l and abs(l) > 0.0001:
            ratio = abs(m/l)
        else:
            ratio = float('inf') if m and m != 0 else 0
        print(f"{feat:<35} {m:>10.3f} {l:>10.3f} {ratio:>8.2f}x")
    except Exception as e:
        print(f"{feat:<35} ERROR: {e}")

All feature files saved.

Feature                                   Mule      Legit    Ratio
-------------------------------------------------------------------
is_frozen                                0.000      0.000     0.00x
account_age_days                       759.000    803.000     0.95x
had_mobile_update                        0.000      0.000     0.00x
days_since_mobile_update                85.000     90.000     0.94x
balance_to_txn_ratio                     0.202      0.196     1.03x
volume_to_balance_ratio               4908.946   5804.314     0.85x
branch_relative_risk                     3.581      0.000      infx
mules_per_employee                       0.050      0.000      infx
kyc_doc_score                            2.000      2.000     1.00x
digital_banking_score                    1.000      1.000     1.00x
is_joint_account                         0.000      0.000     0.00x
is_nri                                   0.000      0.000     0.00x
customer_age_days      

In [ ]:
binary_features = ["is_frozen", "had_mobile_update", "is_joint_account", "is_nri"]

print(f"{'Feature':<35} {'Mule Mean':>12} {'Legit Mean':>12} {'Ratio':>8}")
print("-" * 71)
for feat in binary_features:
    try:
        m = mule_m[feat].mean()
        l = legit_m[feat].mean()
        ratio = m/l if l and l > 0 else float('inf')
        print(f"{feat:<35} {m:>12.4f} {l:>12.4f} {ratio:>8.2f}x")
    except Exception as e:
        print(f"{feat:<35} ERROR: {e}")

# Also check frozen rate properly
print(f"\nFrozen accounts:")
print(f"Mules frozen: {mule_m['is_frozen'].sum()} / {len(mule_m)} = {mule_m['is_frozen'].mean():.1%}")
print(f"Legit frozen: {legit_m['is_frozen'].sum()} / {len(legit_m)} = {legit_m['is_frozen'].mean():.1%}")

Feature                                Mule Mean   Legit Mean    Ratio
-----------------------------------------------------------------------
is_frozen                                 0.4912       0.0449    10.93x
had_mobile_update                         0.1882       0.1513     1.24x
is_joint_account                          0.0857       0.0811     1.06x
is_nri                                    0.0432       0.0309     1.40x

Frozen accounts:
Mules frozen: 1318 / 2683 = 49.1%
Legit frozen: 4197 / 93408 = 4.5%


In [ ]:
contamination = pl.read_parquet(OUTPUTS / "features_contamination.parquet")
entropy = pl.read_parquet(OUTPUTS / "features_entropy.parquet")
passthrough = pl.read_parquet(OUTPUTS / "features_passthrough.parquet")
burst = pl.read_parquet(OUTPUTS / "features_burst.parquet")
txn_derived = pl.read_parquet(OUTPUTS / "features_txn_derived.parquet")
account_feat = pl.read_parquet(OUTPUTS / "features_account.parquet")
branch_feat = pl.read_parquet(OUTPUTS / "features_branch.parquet")
customer_feat = pl.read_parquet(OUTPUTS / "features_customer.parquet")

# Build master — all 160k accounts
master = (
    accounts.select(["account_id", "branch_code"])
    .join(txn_derived, on="account_id", how="left")
    .join(passthrough.select(["account_id", "passthrough_rate", "active_days"]),
          on="account_id", how="left")
    .join(burst.select(["account_id", "burst_ratio", "std_monthly_txn", "active_months"]),
          on="account_id", how="left")
    .join(entropy.select(["account_id", "counterparty_entropy", "unique_cp_count", "pct_volume_top3"]),
          on="account_id", how="left")
    .join(contamination.select(["account_id", "mule_network_cp_count", 
                                 "mule_cp_weighted_score", "contamination_rate",
                                 "max_mule_cp_connection"]),
          on="account_id", how="left")
    .join(account_feat.drop("branch_code"), on="account_id", how="left")
    .join(branch_feat.select(["branch_code", "branch_mule_rate", 
                               "branch_relative_risk", "mules_per_employee",
                               "branch_type_encoded", "branch_turnover",
                               "branch_asset_size"]),
          on="branch_code", how="left")
    .join(linkage, on="account_id", how="left")
    .join(customer_feat, on="customer_id", how="left")
)

# Save full master
master.write_parquet(OUTPUTS / "master_features_all.parquet")
print(f"Master features shape: {master.shape}")
print(f"Columns: {len(master.columns)}")
print(master.columns)

Master features shape: (160153, 80)
Columns: 80
['account_id', 'branch_code', 'total_txn_count', 'avg_amount', 'std_amount', 'total_amount', 'max_amount', 'min_amount', 'reversal_count', 'credit_count', 'debit_count', 'unique_channels', 'unique_counterparties', 'atm_count', 'upi_credit_count', 'upi_debit_count', 'imps_count', 'neft_count', 'standing_instr_count', 'first_txn_date', 'last_txn_date', 'avg_txn_hour', 'late_night_txn_count', 'credit_ratio', 'debit_ratio', 'atm_rate', 'upi_rate', 'interbank_rate', 'standing_instr_rate', 'reversal_rate', 'amount_volatility', 'active_span_days', 'txn_velocity', 'late_night_rate', 'passthrough_rate', 'active_days', 'burst_ratio', 'std_monthly_txn', 'active_months', 'counterparty_entropy', 'unique_cp_count', 'pct_volume_top3', 'mule_network_cp_count', 'mule_cp_weighted_score', 'contamination_rate', 'max_mule_cp_connection', 'account_age_days', 'is_frozen', 'days_since_mobile_update', 'had_mobile_update', 'balance_monthly_quarterly_diff', 'balanc

In [ ]:
# Training set only — with labels
train_master = (
    labels
    .join(master, on="account_id", how="left")
)

train_master.write_parquet(OUTPUTS / "train_features.parquet")
print(f"Train features shape: {train_master.shape}")
print(f"Mules in train: {train_master['is_mule'].sum()}")
print(f"Nulls per column:")
null_counts = train_master.null_count()
# Only show columns with nulls
null_summary = [(col, null_counts[col][0]) for col in null_counts.columns 
                if null_counts[col][0] > 0]
null_summary.sort(key=lambda x: x[1], reverse=True)
for col, count in null_summary[:20]:
    print(f"  {col}: {count} ({count/len(train_master):.1%})")

Train features shape: (96091, 84)
Mules in train: 2683
Nulls per column:
  alert_reason: 93653 (97.5%)
  freeze_duration_days: 93628 (97.4%)
  mule_flag_date: 93408 (97.2%)
  flagged_by_branch: 93408 (97.2%)
  days_since_mobile_update: 81452 (84.8%)
  kyc_doc_score: 40853 (42.5%)
  days_since_passbook_update: 29047 (30.2%)
  balance_monthly_quarterly_diff: 5718 (6.0%)
  balance_daily_monthly_diff: 5718 (6.0%)
  avg_balance: 5718 (6.0%)
  monthly_avg_balance: 5718 (6.0%)
  balance_to_txn_ratio: 5718 (6.0%)
  volume_to_balance_ratio: 5718 (6.0%)
  std_monthly_txn: 2044 (2.1%)
  first_txn_date: 1885 (2.0%)
  last_txn_date: 1885 (2.0%)
  avg_txn_hour: 1885 (2.0%)
  active_span_days: 1885 (2.0%)
  txn_velocity: 1885 (2.0%)


In [ ]:
# Columns to exclude — leakage or redundant
EXCLUDE = [
    "alert_reason", "mule_flag_date", "flagged_by_branch",  # leakage
    "first_txn_date", "last_txn_date",                       # datetime not usable directly
    "account_id", "branch_code", "customer_id",              # identifiers
    "product_family", "rural_branch", "branch_type_encoded", # will encode separately
    "total_amount_right", "avg_amount_right",                 # duplicates
]

# Fill nulls appropriately
train_clean = (
    train_master
    .with_columns([
        # Never updated = very long time ago
        pl.col("days_since_mobile_update").fill_null(9999),
        pl.col("days_since_passbook_update").fill_null(9999),
        pl.col("days_since_address_update").fill_null(9999),
        
        # Never frozen = 0 duration
        pl.col("freeze_duration_days").fill_null(0),
        
        # No KYC docs = 0
        pl.col("kyc_doc_score").fill_null(0),
        
        # No transactions = 0 for all count/rate features
        pl.col("first_txn_date").fill_null(pl.lit(None)),
        pl.col("active_span_days").fill_null(0),
        pl.col("txn_velocity").fill_null(0),
        pl.col("avg_txn_hour").fill_null(12),
        pl.col("std_monthly_txn").fill_null(0),
        pl.col("burst_ratio").fill_null(1),
        
        # Balance nulls — fill with median
        pl.col("avg_balance").fill_null(pl.col("avg_balance").median()),
        pl.col("monthly_avg_balance").fill_null(pl.col("monthly_avg_balance").median()),
        pl.col("balance_monthly_quarterly_diff").fill_null(0),
        pl.col("balance_daily_monthly_diff").fill_null(0),
        pl.col("balance_to_txn_ratio").fill_null(0),
        pl.col("volume_to_balance_ratio").fill_null(0),
    ])
    .drop([c for c in EXCLUDE if c in train_master.columns])
)

# Verify no nulls remain in model features
model_cols = [c for c in train_clean.columns 
              if c not in ["is_mule", "account_id"]]
remaining_nulls = train_clean.select(model_cols).null_count()
null_remaining = [(col, remaining_nulls[col][0]) 
                  for col in remaining_nulls.columns 
                  if remaining_nulls[col][0] > 0]

if null_remaining:
    print("Remaining nulls:")
    for col, count in sorted(null_remaining, key=lambda x: x[1], reverse=True):
        print(f"  {col}: {count}")
else:
    print("No nulls remaining in model features")

print(f"\nFinal train shape: {train_clean.shape}")
train_clean.write_parquet(OUTPUTS / "train_model_ready.parquet")
print("Saved to train_model_ready.parquet")

No nulls remaining in model features

Final train shape: (96091, 71)
Saved to train_model_ready.parquet


In [1]:
from pathlib import Path

print("Train ready:", Path("../outputs/train_model_ready.parquet").exists())
print("MCC features:", Path("../outputs/features_mcc.parquet").exists())
print("Txn features:", Path("../outputs/features_txn_basic.parquet").exists())

Train ready: True
MCC features: True
Txn features: True


In [2]:
print(train.shape)
print(train.columns)
print(train['is_mule'].mean())

NameError: name 'train' is not defined

In [3]:
import polars as pl
from pathlib import Path

OUTPUTS = Path("../outputs")

train = pl.read_parquet(OUTPUTS / "train_model_ready.parquet")

print(train.shape)
print(train.columns)
print(train["is_mule"].mean())

(96091, 72)
['account_id', 'is_mule', 'total_txn_count', 'avg_amount', 'std_amount', 'total_amount', 'max_amount', 'min_amount', 'reversal_count', 'credit_count', 'debit_count', 'unique_channels', 'unique_counterparties', 'atm_count', 'upi_credit_count', 'upi_debit_count', 'imps_count', 'neft_count', 'standing_instr_count', 'avg_txn_hour', 'late_night_txn_count', 'credit_ratio', 'debit_ratio', 'atm_rate', 'upi_rate', 'interbank_rate', 'standing_instr_rate', 'reversal_rate', 'amount_volatility', 'active_span_days', 'txn_velocity', 'late_night_rate', 'passthrough_rate', 'active_days', 'burst_ratio', 'std_monthly_txn', 'active_months', 'counterparty_entropy', 'unique_cp_count', 'pct_volume_top3', 'mule_network_cp_count', 'mule_cp_weighted_score', 'contamination_rate', 'max_mule_cp_connection', 'account_age_days', 'is_frozen', 'days_since_mobile_update', 'had_mobile_update', 'balance_monthly_quarterly_diff', 'balance_daily_monthly_diff', 'freeze_duration_days', 'kyc_flag', 'has_nomination'

## Day 2 — Summary

Behavioural features achieve **OOF AUC ≈ 0.85–0.88**.

Key insight:
- Behavioural statistics capture **how money moves**.
- They cannot capture **who it moves to**.

**Next:** Day3_features.ipynb — build network contamination and anomaly features.